Choix des paramètres

Avant de commencer l'expérience, utilisés l'interface graphique pour :
Définir le deck et les outils utilisés
Imprimer le moule, plaque lasercut, a l'aide du fichier DXF
Choisir les paramètres de l'expériences
placer le fichier JSON dans le dossier A REMPLIR

In [1]:
from science_jubilee.JubileeManager import JubileeManager
from science_jubilee.labware.Labware import *
from science_jubilee.decks.Deck import *

from science_jubilee.decks.LoadAll import *
from science_jubilee.tools.Detection_lentille import Detection_lentille

from science_jubilee.tools.Loop import *
from science_jubilee.tools.HTTPSyringe import *


In [2]:
# Connexion à la Jubilee
jm = JubileeManager(address="10.0.9.6", simulated=False)

detect = Detection_lentille

ser = HTTPSyringe
ino = Loop

2026-04-08 12:14:19 - [JubileeController]  - INFO - Initializing JubileeController (simulated=False, address=10.0.9.6)
2026-04-08 12:14:19 - [JubileeController]  - WARNING - Disconnecting this application from the network will halt connection to Jubilee.
2026-04-08 12:14:19 - [JubileeController]  - INFO - Connecting to Jubilee machine...
2026-04-08 12:14:20 - [JubileeController]  - INFO - Successfully connected and initialized Jubilee machine.
2026-04-08 12:14:20 - [JubileeManager]     - INFO - JubileeManager initialized (simulated=False, max_tools=4).


In [3]:
# Homing
jm.controller.home_all()

Is a tool currently mounted? [y/n]  y
Are you ready to remove it now? [y/n]  y


Resuming homing process.


Is the deck clear of any obstacles? [y/n]  n


2026-04-08 11:31:09 - [JubileeController]  - INFO - Homing cancelled by user (deck not clear).


Please clear the deck before homing the Z axis.


In [3]:
# Enregistrer le deck
deck = jm.load_deck("test1")
load_all(jm,"test1")

#affiche les slots
jm.deck.list_slots()
jm.deck.get_all_slot_machine_coordinates()

jm.status()

# Descendre le plateau pour positionner la feuille sur laquel sera placer le plan
jm.controller.move_to(z=200)
jm.controller.move_to(x=150, y=150)

2026-04-08 12:14:25 - [Deck]               - INFO - Loading deck configuration from: /home/sworkyx/Documents/Polytech/Projet_indus/science-jubilee/src/science_jubilee/decks/deck_definition/test1.json
2026-04-08 12:14:25 - [Deck]               - INFO - Deck 'Experience1' loaded with 3 slots.
2026-04-08 12:14:25 - [JubileeManager]     - INFO - Deck 'Experience1' loaded from '/home/sworkyx/Documents/Polytech/Projet_indus/science-jubilee/src/science_jubilee/decks/deck_definition'.
2026-04-08 12:14:25 - [Tool]               - INFO - Tool 'Inoculator' (index 0) initialized.
2026-04-08 12:14:25 - [JubileeManager]     - INFO - Tool 'Inoculator' loaded at index 0.
2026-04-08 12:14:25 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, 36, 0).
2026-04-08 12:14:25 - [Tool]               - INFO - Tool 'Pipette' (index 1) initialized.
2026-04-08 12:14:25 - [JubileeManager]     - INFO - Tool 'Pipette' loaded at index 1.
2026-04-08 12:14:25 - [JubileeManager]     - INFO - Offset for

In [5]:
dict_outil = jm.get_tool_by_name("stylo")
idx = dict_outil['index']

jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 35))


2026-04-08 12:14:47 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 2.
2026-04-08 12:14:54 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 2.
2026-04-08 12:14:54 - [JubileeManager]     - INFO - Tool at index 2 set as active tool.
2026-04-08 12:14:54 - [JubileeManager]     - INFO - Offset for tool at index 2 set to (0, -31.5, 35).


In [6]:
execute_plan('plan_jubilee', jm=jm)
jm.controller.move_to(z=200)
jm.controller.move_to(x=150, y=150)

🚀 Exécution du plan Jubilee : plan_jubilee.txt
✅ Plan terminé avec succès sur Jubilee.


Préparation de l'expérience

- Placer le deck d'expérience
- Placer les outils nécéssaires

In [6]:
jm.controller.park_tool_sequence(idx)

# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Pipette")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 35))

tool1 = jm.tools_list[idx]
tool1.__class__ = ser

tool1.url_materiel = "http://10.0.9.55:5001" 
tool1._init_gpio()

tool1._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool1.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool1.safe_z_movement = lambda: jm.controller.move_to(z=220)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool1.name}' prêt (Classe: {type(tool1).__name__})")

slot_1 = jm.deck.get_slot('1')
well_eau = slot_1.labware.wells['A1']
well_eau.y = well_source.y + slot_1.coordinates[1] - 50 # ac offset de x
well_eau.x = well_source.x + slot_1.coordinates[0] - 5.5 #ac offset de y



2026-04-08 12:15:49 - [JubileeController]  - INFO - Starting park_tool sequence for tool 2.
2026-04-08 12:15:54 - [JubileeController]  - INFO - park_tool_sequence completed for tool 2.
2026-04-08 12:15:54 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 1.
2026-04-08 12:16:00 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 1.
2026-04-08 12:16:00 - [JubileeManager]     - INFO - Tool at index 1 set as active tool.
2026-04-08 12:16:00 - [JubileeManager]     - INFO - Offset for tool at index 1 set to (0, -31.5, 35).


[Pipette] ✅ Connecté au serveur matériel du Raspberry Pi.
✅ Correctif safe_z_movement appliqué.
✅ Outil 'Pipette' prêt (Classe: HTTPSyringe)


NameError: name 'well_source' is not defined

In [8]:
# offset pipette
# x = -5.5
# y = -50
# z = 118 pour vider

cpt = 0 
# --- CONFIGURATION DES PUITS ---
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# On définit la destination
slot_0 = jm.deck.get_slot('0')

jm.controller.move_to(z=220)
jm.controller.move_to(x=150, y=150)

print(f"🚀 Début de la distribution depuis {well_eau.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        if ( cpt == 0 ):
            jm.controller.move_to(z=220)
            jm.controller.move_to(x=well_eau.x,y=well_eau.y)
            jm.controller.move_to(z=120)
            try:
                jm.controller.dwell(1000)
                tool1.remplir_seringue(temps_secondes=10.0)
            except Exception as e:
                print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
                break
            jm.controller.move_to(z=220)
            cpt = 3
            
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6
        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]
        well_dest.y = (well_dest.y + slot_0.coordinates[1]) - 50
        well_dest.x = (well_dest.x + slot_0.coordinates[0]) - 5.5

        print(f"\n---> Transfert de {well_eau.name} vers {well_dest_name}...")

        jm.controller.move_to(x=well_dest.x,y=well_dest.y)
        jm.controller.move_to(z=118)
        try:
            jm.controller.dwell(1000)
            tool1.avancer_jusqu_au_seuil(seuil=0.6, timeout_sec=3.333333333)
        except Exception as e:
            print(f"❌ Erreur sur le puits {well_dest_name} : {e}")
            break
        cpt = cpt - 1
        jm.controller.move_to(z=130)

print("\n✅ Distribution terminée sur toute la plaque !")

🚀 Début de la distribution depuis A1...
[Pipette] Vidage en cours pour 4.0 sec...


KeyboardInterrupt: 

In [7]:
jm.controller.park_tool_sequence(idx)

# 1. On récupère les infos de ton outil par son nom
dict_outil = jm.get_tool_by_name("Inoculator")
idx = dict_outil['index']

# 2. On prépare l'outil physiquement
jm.controller.pickup_tool_sequence(idx)
jm.set_active_tool(idx)
jm.set_tool_offset(idx, (0, -31.5, 68))

tool0 = jm.tools_list[idx]
tool0.__class__ = Loop
tool0._machine = jm.controller  # On le lie au contrôleur pour les mouvements
tool0.is_active_tool = True     # On valide le décorateur de sécurité

# 4. CORRECTIF : On crée la méthode safe_z_movement si elle manque
if not hasattr(jm.controller, 'safe_z_movement'):
    # On remonte à 80mm de hauteur pour ne rien toucher
    tool0.safe_z_movement = lambda: jm.controller.move_to(z=80)
    print("✅ Correctif safe_z_movement appliqué.")

print(f"✅ Outil '{tool0.name}' prêt (Classe: {type(tool0).__name__})")

slot_1 = jm.deck.get_slot('1')
well_source = slot_1.labware.wells['A1'] ;
well_lentille.y = well_source.y - 31 + slot_1.coordinates[1] 
well_lentille.x = well_source.x + slot_1.coordinates[0] -7  #maj x


2026-04-08 12:06:59 - [JubileeController]  - INFO - Starting park_tool sequence for tool 1.
2026-04-08 12:07:06 - [JubileeController]  - INFO - park_tool_sequence completed for tool 1.
2026-04-08 12:07:06 - [JubileeController]  - INFO - Starting pickup_tool sequence for tool 0.
2026-04-08 12:07:12 - [JubileeController]  - INFO - pickup_tool_sequence completed for tool 0.
2026-04-08 12:07:12 - [JubileeManager]     - INFO - Tool at index 0 set as active tool.
2026-04-08 12:07:12 - [JubileeManager]     - INFO - Offset for tool at index 0 set to (0, -31.5, 68).


✅ Correctif safe_z_movement appliqué.
✅ Outil 'Inoculator' prêt (Classe: Loop)


In [8]:
rows = ['A', 'B', 'C', 'D']
cols = range(1, 7)

# On définit la destination fixe
slot_0 = jm.deck.get_slot('0')
jm.controller.move_to(z=200)

print(f"🚀 Début de la distribution depuis {well_source.name}...")

# --- LA BOUCLE DE DISTRIBUTION ---
for r in rows:
    for c in cols:
        well_dest_name = f"{r}{c}" # Génère A1, A2... D6

        
        # On récupère l'objet Well de destination
        well_dest = slot_0.labware.wells[well_dest_name]

        well_dest.x = (well_dest.x + slot_0.coordinates[0] -7 )
        well_dest.y = (well_dest.y- 41 + slot_0.coordinates[1])#-41 ici car transferyt utilise move_to et pas effecteur to
        print(well_dest)

        print(f"Transfert de {well_source.name} vers {well_dest_name}...")

        try:
                # Appel du transfert (la sécurité safe_z est déjà injectée)
                jm.controller.move_to(z=200)        
                tool0.transfer(
                    source=well_lentille,      # FIXE
                    destination=well_dest,   # CHANGE à chaque itération
                    s=2000,
                    sweep_x=3,
                    sweep_y=3,
                    sweep_z=10,
                    sweep_speed=150,
                    up_speed=800,
                    randomize_pickup=True
                )
        except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break
                
        jm.controller.move_to(z=90)
        jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
        jm.controller.dwell(1000)
        detect.capture_octopi_image()
        jm.controller.dwell(1000)

        compteur = 0      
        while(detect.detect_lens() == False and compteur < 3 ):
            compteur= compteur + 1
            jm.controller.move_to(z=200)
            try:
                    # Appel du transfert (la sécurité safe_z est déjà injectée)
                tool0.transfer(
                source=well_lentille,      # FIXE
                destination=well_dest,   # CHANGE à chaque itération
                s=2000,
                sweep_x=3,
                sweep_y=3,
                sweep_z=10,
                sweep_speed=150,
                up_speed=800,
                randomize_pickup=True
                    )
            except Exception as e:
                print(f"❌ Erreur critique au puits {well_dest_name}: {e}")
                # Sécurité : on remonte immédiatement
                jm.controller.move_to(z=200)
                break 
            jm.controller.move_to(z=90)
            jm.controller.move_to(x=well_dest.x, y=well_dest.y+36+20)
            jm.controller.dwell(5000)
            detect.capture_octopi_image()
            jm.controller.dwell(1000)


jm.controller.park_tool_sequence(idx)


  


print("✅ Distribution terminée sur toute la plaque !")

🚀 Début de la distribution depuis A1...
Well A1 at coordinates (408.74, -44.14, 2.5)
Transfert de A1 vers A1...


2026-04-08 12:08:01 - [JubileeController]  - ERROR - Absolute move exceeds Y axis limit (-44.0–345.0 mm)
2026-04-08 12:08:01 - [JubileeController]  - ERROR - State error during move_to: Absolute move exceeds Y axis limit (-44.0–345.0 mm)


❌ Erreur critique au puits A1: Absolute move exceeds Y axis limit (-44.0–345.0 mm)
Well B1 at coordinates (428.24, -44.14, 2.5)
Transfert de A1 vers B1...


2026-04-08 12:08:02 - [JubileeController]  - ERROR - Absolute move exceeds Y axis limit (-44.0–345.0 mm)
2026-04-08 12:08:02 - [JubileeController]  - ERROR - State error during move_to: Absolute move exceeds Y axis limit (-44.0–345.0 mm)


❌ Erreur critique au puits B1: Absolute move exceeds Y axis limit (-44.0–345.0 mm)
Well C1 at coordinates (447.74, -44.14, 2.5)
Transfert de A1 vers C1...


...

❌ Erreur critique au puits C1: Absolute move exceeds Y axis limit (-44.0–345.0 mm)
Well D1 at coordinates (467.24, -44.14, 2.5)
Transfert de A1 vers D1...


2026-04-08 12:08:04 - [JubileeController]  - ERROR - Absolute move exceeds Y axis limit (-44.0–345.0 mm)
2026-04-08 12:08:04 - [JubileeController]  - ERROR - State error during move_to: Absolute move exceeds Y axis limit (-44.0–345.0 mm)


❌ Erreur critique au puits D1: Absolute move exceeds Y axis limit (-44.0–345.0 mm)


2026-04-08 12:08:04 - [JubileeController]  - INFO - Starting park_tool sequence for tool 0.
2026-04-08 12:08:10 - [JubileeController]  - INFO - park_tool_sequence completed for tool 0.


✅ Distribution terminée sur toute la plaque !


AttributeError: 'NoneType' object has no attribute 'url_materiel'